In [5]:
import numpy as np
import pandas as pd
import torch
from scipy.optimize import linprog, minimize, LinearConstraint

In [ ]:
# calculate discount factor D, forward price F
# Call − Put = D·(F − K)
def implied_D_F(chain):

    c = chain[chain.cp_flag == "C"].groupby("K").agg(cb=("best_bid", "first"), ca=("best_offer", "first"))
    p = chain[chain.cp_flag == "P"].groupby("K").agg(pb=("best_bid", "first"), pa=("best_offer", "first"))
    m = c.join(p, how="outer").reset_index()

    # keep only strikes where all 4 quotes are real
    liq = m[(m.cb > 0) & (m.pb > 0) & (m.ca > 0) & (m.pa > 0)]
    # too little data: don't include this expiry
    if len(liq) < 3:
        return None
    
    K = liq.K.values
    # lowest possible Call-Put  (buy call at bid, sell put at ask)
    L = (liq.cb - liq.pa).values
    # highest possible Call-Put
    U = (liq.ca - liq.pb).values

    # fit a straight line A − D·K that stays inside every [L,U]
    A_ub, b_ub = [], []
    for i in range(len(K)):
        #  -(A - D*K) + t <= -L (lower bound)
        A_ub.append([-1.0, -K[i], 1.0])
        b_ub.append(-L[i])

        #   (A - D*K) + t <= U (upper bound)
        A_ub.append([ 1.0,  K[i], 1.0])
        b_ub.append( U[i])
    
    # variables: [A, D, t]
    # maximizing t (the -1 in the objective) finds the line that fits the parity relation with the most slack
    res = linprog([0.0, 0.0, -1.0], A_ub=A_ub, b_ub=b_ub,
                  bounds=[(None, None)] * 3, method="highs")
    if res.success:
        A, B, t = res.x
    else:
        # if LP fails, fall back to a least-squares line fit.
        cmid = 0.5 * (liq.cb + liq.ca)
        pmid = 0.5 * (liq.pb + liq.pa)
        B, A = np.polyfit(K, (cmid - pmid).values, 1)
        t = float("nan")
    D = -B
    F = A / D
    if not (0 < D < 1.2 and F > 0):
        return None
    return D, F, m, float(t)

# print per-call analytic-center diagnostics
ACEN_DEBUG   = True

# Newton iteration cap (replaces trust-constr maxiter)
ACEN_MAXITER = 100

# finds the exact analytic center (the "most balanced" interior point) using Newton's method on -Σ log(slack)
# log-barrier  f(x) = -sum log(b - M x)
def _analytic_center_newton(M, b, x0, max_iter=ACEN_MAXITER, tol=1e-10):
    x = np.asarray(x0, float).copy()
    n = len(x); nit = 0
    for nit in range(1, max_iter + 1):
        s = b - M @ x
        if np.any(s <= 0):
            break
        inv = 1.0 / s
        g = M.T @ inv                               # gradient
        H = (M * (inv * inv)[:, None]).T @ M        # M^T diag(1/s^2) M  (SPD)
        H[np.diag_indices(n)] += 1e-12 * (np.trace(H) / n + 1.0)   # tiny ridge
        try:
            dx = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            dx, *_ = np.linalg.lstsq(H, -g, rcond=None)
        dec = float(-g @ dx)                        # Newton decrement^2 (>= 0)
        if dec <= tol:
            break
        Mdx = M @ dx                                # longest step keeping s > 0
        pos = Mdx > 0
        amax = np.min(s[pos] / Mdx[pos]) if np.any(pos) else 1.0
        a = min(1.0, 0.99 * amax)
        f0 = -np.sum(np.log(s))
        while a > 1e-14:                            # feasible Armijo backtrack
            sn = b - M @ (x + a * dx)
            if np.all(sn > 0) and -np.sum(np.log(sn)) <= f0 - 1e-4 * a * dec:
                break
            a *= 0.5
        x = x + a * dx
        if a <= 1e-14:
            break
    return x, nit


def arb_free_analytic_center(K, B, A, S0):

    # Solver: Chebyshev-center LP (strictly-interior start) -> damped Newton.
    K  = np.asarray(K, float)
    Ch = np.asarray(A, float)
    Cl = np.asarray(B, float)
    N  = len(K)
    L    = 0.0
    Kbar = max(2000.0, float(K[-1]) + 100.0)

    lb = np.maximum.reduce([np.zeros(N), S0 - K, Cl])
    ub = np.minimum(np.full(N, S0), Ch)

    def _base():
        M = np.zeros((N - 1 + N - 2, N)); bb = np.zeros(N - 1 + N - 2)

        # decreasing
        for i in range(N - 1):
            M[i, i] = -1.0; M[i, i + 1] = 1.0

        # convex
        for i in range(1, N - 1):
            M[N - 1 + i - 1, i - 1] = -1.0 / (K[i] - K[i - 1])
            M[N - 1 + i - 1, i]     =  1.0 / (K[i] - K[i - 1]) + 1.0 / (K[i + 1] - K[i])
            M[N - 1 + i - 1, i + 1] = -1.0 / (K[i + 1] - K[i])
        return M, bb

    def _phantom():
        M, bb = _base()
        E = np.zeros((4, N)); eb = np.zeros(4)
        # row E[0]: 
        # slope >= -1, (C[0] − S0)/(K[0] − L) ≥ −1
        # -C[0] ≤ K[0] − L − S0
        E[0, 0]   = -1.0
        eb[0] = K[0] - L - S0

        # row E[1]: C_{N-1} >= 0
        E[1, N-1] = -1.0
        eb[1] = 0.0

        # row E[2]: convexity at K[0]
        # C[0]·(1/(K[0]−L) + 1/(K[1]−K[0]))  −  C[1]/(K[1]−K[0])  ≤  S0/(K[0]−L)
        E[2, 0]   = 1.0/(K[0]-L) + 1.0/(K[1]-K[0])
        E[2, 1]   = -1.0/(K[1]-K[0])
        eb[2] = S0/(K[0]-L)

        # row E[3]: convexity at K[N-1]
        # C[N-1]·(1/(Kbar−K[N-1]) + 1/(K[N-1]−K[N-2]))  −  C[N-2]/(K[N-1]−K[N-2])  ≤  0
        E[3, N-2] = -1.0/(K[N-1]-K[N-2])
        E[3, N-1] = 1.0/(Kbar-K[N-1]) + 1.0/(K[N-1]-K[N-2])
        eb[3] = 0.0

        return np.vstack([M, E]), np.concatenate([bb, eb])

    def _legacy():
        # no additional points at C(0)=S0, C(Kbar)=0
        M, bb = _base()
        E = np.zeros((2, N))
        eb = np.zeros(2)

        # E[0]:  (C[0]−C[1])/(K[1]−K[0]) ≤ 1
        # first slope >= -1
        E[0, 0] =  1.0/(K[1]-K[0]); E[0, 1] = -1.0/(K[1]-K[0]); eb[0] = 1.0

        # E[1]:  (C[N-1]−C[N-2])/(K[N-1]−K[N-2]) ≤ 0
        # last slope <= 0
        E[1, N-2] = -1.0/(K[N-1]-K[N-2]); E[1, N-1] = 1.0/(K[N-1]-K[N-2]); eb[1] = 0.0
        return np.vstack([M, E]), np.concatenate([bb, eb])

    # Chebyshev center = center of the largest ball that fits inside the polytope
    def _cheb_solve(M, bb):
        # augment with box rows, then Chebyshev-center LP for a strictly-interior x0
        Mc = np.vstack([M, np.eye(N), -np.eye(N)])
        bc = np.concatenate([bb, ub, -lb])
        rn = np.linalg.norm(Mc, axis=1)
        Ac = np.hstack([Mc, rn[:, None]])

        # maximize inscribed radius r
        cc = np.zeros(N + 1); cc[-1] = -1.0
        res = linprog(cc, A_ub=Ac, b_ub=bc, bounds=[(None, None)] * N + [(0, None)], method="highs")
        return Mc, bc, res

    M, b, res = _cheb_solve(*_phantom())
    scheme = "phantom"
    if not res.success:
        M, b, res = _cheb_solve(*_legacy())
        scheme = "legacy (phantom LP infeasible)"

    if res.success:
        x0  = res.x[:N]; rad = float(res.x[-1])
    else:
        x0  = np.clip(0.5 * (Cl + Ch), lb, ub); rad = 0.0
    if ACEN_DEBUG:
        print(f"      [acen] N={N} scheme={scheme} LP_ok={res.success} cheb_r={rad:.3g} "
              f"-> Newton centering ...", flush=True)

    import time as _time
    _t = _time.time()
    if rad > 1e-9 and np.all(b - M @ x0 > 0):
        x, nit = _analytic_center_newton(M, b, x0)
    else:
        # empty interior -> use Chebyshev/LP point
        x, nit = x0, 0
        if ACEN_DEBUG:
            print(f"      [acen] N={N} WARNING: empty interior (cheb_r={rad:.2g}); the convex "
                  f"in-band polytope is degenerate -- returning a non-centered (possibly "
                  f"non-convex) point.  Widen the band or relax constraints for this expiry.",
                  flush=True)
    if ACEN_DEBUG:
        print(f"      [acen] N={N} center done: newton_iters={nit} t={_time.time()-_t:.2f}s", flush=True)
    return x

In [ ]:
def process_all(csv_path, date, quotes_out=None,
                arbfree_out=None, min_strikes=8, verbose=True):
    # process every expiry of one trade date
    if quotes_out  is None: 
        quotes_out  = f"Quotes_SPX_{date}.csv"
    if arbfree_out is None: 
        arbfree_out = f"ArbFree_SPX_{date}.csv"

    df = pd.read_csv(csv_path)
    df = df[df["date"] == date].copy()
    df["K"] = df["strike_price"] / 1000.0
    perexp = {}
    for exdate, chain in df.groupby("exdate"):
        r = implied_D_F(chain)
        if r is None:
            continue
        D, F, m, t = r
        T = np.busday_count(np.datetime64(pd.to_datetime(str(date),   format="%Y%m%d").date()),
                            np.datetime64(pd.to_datetime(str(exdate), format="%Y%m%d").date())) / 252.0
        if T > 0:
            perexp[int(exdate)] = (D, F, T, m, t)
    if not perexp:
        print("No usable expiries on", date)
        return None
    Ts  = np.array([v[2] for v in perexp.values()])
    DFs = np.array([v[0] * v[1] for v in perexp.values()])

    if len(Ts) >= 2:
        # log(D*F) is ~linear in T
        slope, intercept = np.polyfit(Ts, np.log(DFs), 1)
        # S0 = value at T=0; q = dividend yield
        S0, q = float(np.exp(intercept)), float(-slope)
    else:
        S0, q = float(DFs[0]), 0.0

    qrows, arows = [], []
    for exdate, (D, F, T, m, t) in sorted(perexp.items()):
        mfac = S0 / (D * F)

        # rescale strikes to the S0 measure
        Khat = m.K.values * S0 / F
        getq = lambda col, miss: np.where(np.nan_to_num(col, nan=0.0) > 0, np.nan_to_num(col, nan=0.0), miss)

        # missing ask -> F
        ca = getq(m.ca.values, F)
        pa = getq(m.pa.values, F)

        # missing bid -> 0
        cb = getq(m.cb.values, 0.0)
        pb = getq(m.pb.values, 0.0)

        intr = np.maximum(S0 - Khat, 0.0)

        # tightest ask from call OR put (parity)
        A = np.minimum(mfac * ca, mfac * pa + (S0 - Khat))
        # loosest bid, but >= intrinsic
        B = np.maximum.reduce([mfac * cb, mfac * pb + (S0 - Khat), intr])
        # drop strikes where band is empty
        ok = (A >= B)
        Khat, A, B = Khat[ok], A[ok], B[ok]

        # sort by strike
        o = np.argsort(Khat)
        Khat, A, B = Khat[o], A[o], B[o]
        keep = np.concatenate([[True], np.diff(Khat) > 1e-9])
        Khat, A, B = Khat[keep], A[keep], B[keep]
        if len(Khat) < min_strikes:
            continue
        if verbose:
            print(f"  exp {exdate} | {len(Khat):3d} strikes | solving arb-free ...", flush=True)
        import time as _time
        _t0 = _time.time()

        # arb-free prices
        arb = arb_free_analytic_center(Khat, B, A, S0)
        _acen_dt = _time.time() - _t0
        for k, a, b, c in zip(Khat, A, B, arb):
            qrows.append((date, exdate, k, a, b, round(S0, 4), round(T, 6), round(D, 6), round(F, 4)))
            arows.append((date, exdate, c, k))
        if verbose:
            inb = float(((arb >= B - 1e-6) & (arb <= A + 1e-6)).mean())
            print(f"  exp {exdate} | T={T:.4f} D={D:.5f} F={F:.2f} | {len(Khat):3d} strikes | "
                  f"arb in-band {inb*100:.0f}% | {_acen_dt:.1f}s", flush=True)

    pd.DataFrame(qrows, columns=["date", "exdate", "strike", "ask", "bid", "S0", "T", "D", "F"]).to_csv(quotes_out, index=False)
    pd.DataFrame(arows, columns=["date", "exdate", "price", "strike"]).to_csv(arbfree_out, index=False)
    if verbose:
        print(f"\\nSpot S0 = {S0:.3f} (implied q = {q:.4f}) -> wrote '{quotes_out}' and '{arbfree_out}'")
    return dict(S0=S0, q=q)

# moneyness=(0, np.inf) to keep all strikes
def extract(date, exdate, moneyness=(0, np.inf),
            quotes_file=None, arbfree_file=None):
    if quotes_file is None: 
        quotes_file  = f"Quotes_SPX_{date}.csv"
    if arbfree_file is None: 
        arbfree_file = f"ArbFree_SPX_{date}.csv"

    q = pd.read_csv(quotes_file)
    q = q[(q.date == date) & (q.exdate == exdate)].sort_values("strike")
    
    a = pd.read_csv(arbfree_file)
    a = a[(a.date == date) & (a.exdate == exdate)].sort_values("strike")
    if len(q) == 0:
        raise ValueError(f"no rows for date={date}, exdate={exdate} (run process_all first)")
    S0 = float(q.S0.iloc[0])
    Kbar = float(max(2000.0, float(a.strike.values[-1]) + 100.0))  # match arb_free_analytic_center
    q = q[(q.strike >= moneyness[0] * S0) & (q.strike <= moneyness[1] * S0)]
    a = a[(a.strike >= moneyness[0] * S0) & (a.strike <= moneyness[1] * S0)]
    K = q.strike.values
    bid_ask = torch.tensor(np.column_stack([K, q.bid.values, q.ask.values]), dtype=torch.float64)
    return dict(bid_ask=bid_ask,
                arb_p=torch.tensor(a.price.values, dtype=torch.float64),
                arb_K=torch.tensor(a.strike.values, dtype=torch.float64),
                S0=S0, Kbar=Kbar, T=float(q["T"].iloc[0]), D=float(q.D.iloc[0]), F=float(q.F.iloc[0]),
                R1=int((K <= S0).sum()), R2=int((K >= S0).sum()))

In [8]:
process_all("SPX_opt_2011_2012.csv", 20110103, quotes_out="Quotes_SPX_20110103.csv", arbfree_out="ArbFree_SPX_20110103.csv")

ch = extract(20110103, 20110122)
bid_ask, arb_p, arb_K = ch["bid_ask"], ch["arb_p"], ch["arb_K"]
S0, T, D, F, R1, R2 = ch["S0"], ch["T"], ch["D"], ch["F"], ch["R1"], ch["R2"]
r = -np.log(D) / T
print(f"extracted: S0={S0:.3f}  T={T:.4f}  D={D:.5f}  r={r:+.4f}  R1={R1} R2={R2}  ({len(arb_K)} strikes)")

  exp 20110107 |  31 strikes | solving arb-free ...
      [acen] N=31 scheme=phantom LP_ok=True cheb_r=0.00592 -> Newton centering ...
      [acen] N=31 center done: newton_iters=9 t=0.00s
  exp 20110107 | T=0.0159 D=1.00167 F=1270.74 |  31 strikes | arb in-band 100% | 0.0s
  exp 20110122 | 159 strikes | solving arb-free ...
      [acen] N=159 scheme=phantom LP_ok=True cheb_r=3.59e-05 -> Newton centering ...
      [acen] N=159 center done: newton_iters=18 t=0.04s
  exp 20110122 | T=0.0595 D=1.00045 F=1270.70 | 159 strikes | arb in-band 100% | 0.0s
  exp 20110219 | 156 strikes | solving arb-free ...
      [acen] N=156 scheme=phantom LP_ok=True cheb_r=0.000266 -> Newton centering ...
      [acen] N=156 center done: newton_iters=16 t=0.02s
  exp 20110219 | T=0.1389 D=0.99778 F=1269.01 | 156 strikes | arb in-band 100% | 0.0s
  exp 20110319 | 123 strikes | solving arb-free ...
      [acen] N=123 scheme=phantom LP_ok=True cheb_r=0.000378 -> Newton centering ...
      [acen] N=123 center done